In [33]:
import sys
#!pip install XXX --target ./my_custom_packages
sys.path.append('./my_custom_packages')
import polars as pl
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import accuracy_score


In [9]:
annot_data = {}

with open("GSE148375_series_matrix.txt", "r") as f:
    line = "X"
    while line:
        line = f.readline()
        if "!Sample_" in line:
            content = line.strip().split("\t")
            
            c = content[0]
            
            if c in annot_data.keys():
                i = 2
                c += "_"
                while c + str(i) in annot_data.keys():
                    i += 1
                c += str(i)

                annot_data[c] = content[1:]
            else:
                annot_data[c] = content[1:]
    


annot_data = pl.from_dict(annot_data)



annot_data = annot_data.with_columns(
    Sample = pl.col("!Sample_description").str.replace_all(r'"', "").cast(pl.Int32),
    Status = pl.col("!Sample_characteristics_ch1_7").str.replace_all(r'"', "").str.split(":").list.get(1).str.strip_chars(),
    Age = pl.col("!Sample_characteristics_ch1_2").str.replace_all(r'"', "").str.split(":").list.get(1).str.strip_chars().cast(pl.Int8),
    Gender = pl.col("!Sample_characteristics_ch1_3").str.replace_all(r'"', "").str.split(":").list.get(1).str.strip_chars() == "Male",
    CPD = pl.col("!Sample_characteristics_ch1_4").str.replace_all(r'"', "").str.split(":").list.get(1).str.strip_chars().cast(pl.Int8),
    HSI = pl.col("!Sample_characteristics_ch1_5").str.replace_all(r'"', "").str.split(":").list.get(1).str.strip_chars().cast(pl.Int8),
    FTND = pl.col("!Sample_characteristics_ch1_6").str.replace_all(r'"', "").str.split(":").list.get(1).str.strip_chars().cast(pl.Int8),
    

)

annot_data = annot_data.select(["Sample", "Status", "Age", "Gender", "CPD", "HSI", "FTND"])



display(annot_data.head())






Sample,Status,Age,Gender,CPD,HSI,FTND
i32,str,i8,bool,i8,i8,i8
200026,"""Smoker""",39,true,20,4,7
200027,"""Smoker""",42,true,30,5,9
200028,"""Smoker""",32,false,40,6,9
200032,"""Smoker""",33,true,20,4,7
200033,"""Smoker""",48,false,10,3,5
…,…,…,…,…,…,…
200371,"""Non-smoker""",24,false,-9,-9,-9
200382,"""Smoker""",17,true,5,0,1
200384,"""Non-smoker""",16,false,-9,-9,-9


In [41]:
cost_matrix = pl.read_parquet("cost_matrix_processed.parquet").transpose(include_header=True, header_name="Sample")



cost_matrix = cost_matrix.with_columns([
    pl.col("Sample").cast(pl.Int32)
])



mdf = cost_matrix.join(annot_data, on = "Sample")

mdf = mdf.filter(pl.col("Status").is_in(["Smoker", "Non-smoker"]))





In [54]:
display(mdf.head())

y = mdf.select('Status')

x = mdf.select(pl.all().exclude(['Sample', 'Status']))


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 4, stratify = y)



cols_to_scale = ["Age", "CPD", "HSI", "FTND"]

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(x_train.select(cols_to_scale))
test_scaled = scaler.transform(x_test.select(cols_to_scale))

x_train = x_train.with_columns([
    pl.Series(name, train_scaled[:, i]) for i, name in enumerate(cols_to_scale)
])

x_test = x_test.with_columns([
    pl.Series(name, test_scaled[:, i]) for i, name in enumerate(cols_to_scale)
])




Sample,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,column_14,column_15,column_16,column_17,column_18,column_19,column_20,column_21,column_22,column_23,column_24,column_25,column_26,column_27,column_28,column_29,column_30,column_31,column_32,column_33,column_34,column_35,…,column_203385,column_203386,column_203387,column_203388,column_203389,column_203390,column_203391,column_203392,column_203393,column_203394,column_203395,column_203396,column_203397,column_203398,column_203399,column_203400,column_203401,column_203402,column_203403,column_203404,column_203405,column_203406,column_203407,column_203408,column_203409,column_203410,column_203411,column_203412,column_203413,column_203414,column_203415,Status,Age,Gender,CPD,HSI,FTND
i32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,str,i8,bool,i8,i8,i8
200026,1,2,1,2,2,2,0,0,1,2,1,1,1,1,1,1,0,0,0,1,2,1,1,1,2,2,2,1,2,1,1,0,1,2,1,0,…,2,0,2,2,2,2,2,2,2,2,2,0,2,2,2,2,0,2,0,2,2,2,2,2,2,2,2,2,2,2,2,"""Smoker""",39,true,20,4,7
200027,1,2,1,0,1,2,0,0,2,2,0,1,2,1,2,2,2,2,0,1,0,1,2,2,2,1,2,2,1,2,1,2,2,2,2,0,…,2,0,2,2,2,2,2,2,2,2,2,0,2,2,2,2,0,2,0,2,2,2,2,2,2,2,2,2,2,2,2,"""Smoker""",42,true,30,5,9
200028,1,2,2,2,2,1,0,0,1,2,2,2,1,1,1,2,1,2,1,1,0,1,1,2,2,1,2,0,1,2,0,1,2,2,0,0,…,2,0,2,2,2,2,2,2,2,2,2,0,2,2,2,2,0,2,0,2,2,2,2,2,2,2,2,2,2,2,2,"""Smoker""",32,false,40,6,9
200032,1,1,2,2,1,2,0,0,2,2,1,1,1,0,2,1,2,1,1,2,0,2,2,0,2,1,2,0,1,1,2,1,2,2,1,0,…,2,0,2,2,2,2,2,2,2,2,2,0,2,2,2,2,0,2,0,2,2,2,2,2,2,2,2,2,2,2,2,"""Smoker""",33,true,20,4,7
200033,2,2,1,1,2,2,0,0,2,2,1,2,1,1,2,1,2,2,0,1,0,1,1,0,2,2,2,1,1,1,1,0,2,2,2,0,…,2,0,2,2,2,2,2,2,2,2,2,0,2,2,2,2,0,2,0,2,2,2,2,2,2,2,2,2,2,2,2,"""Smoker""",48,false,10,3,5


In [58]:

t = SelectKBest(score_func = chi2, k = 6500)

t_x_train = t.fit_transform(x_train, y_train)
t_x_test = t.transform(x_test)




lambda_to_try = np.logspace(-4, 2, 100)

#cv_strategy =  StratifiedKFold(n_splits = 10, shuffle = True, random_state = 4)

lasso = LogisticRegressionCV(
  #  cv = cv_strategy,
    Cs = 1/lambda_to_try,
    random_state = 4,
    scoring = "roc_auc",
    penalty = "l1",
    n_jobs = 50,
    max_iter = 10000,
    solver = "liblinear"
)


lasso.fit(t_x_train, y_train)

significant_features = lasso.coef_[0] != 0

x_train_significant = t_x_train[:, significant_features]
x_test_significant = t_x_test[:, significant_features]



print(x_train_significant.shape[1])


/opt/jupyterhub/pyvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


1647


In [59]:

lr = LogisticRegression()

lr.fit(x_train_significant, y_train)

predictions = lr.predict(x_test_significant)
probas = lr.predict_proba(x_test_significant)

print(f"Accuracy score: {accuracy_score(y_test, predictions):.2f}")



Accuracy score: 0.93


/opt/jupyterhub/pyvenv/lib/python3.12/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [62]:
x = mdf.select(pl.all().exclude(['Sample', 'Status', 'Age', 'Gender', 'CPD', 'HSI', 'FTND']))
y = mdf.select('Status')
'''
# 2. Fix the negative numbers BEFORE feature selection
# This pushes the -9 values to 0, preventing the chi2 crash
cols_to_scale = ["Age", "CPD", "HSI", "FTND"]
scaler = MinMaxScaler()

scaled_arrays = scaler.fit_transform(x.select(cols_to_scale))

x = x.with_columns([
    pl.Series(name, scaled_arrays[:, i]) for i, name in enumerate(cols_to_scale)
])'''

t = SelectKBest(score_func=chi2, k=6500)
x_cheating = t.fit_transform(x.to_numpy(), np.ravel(y)) 

x_train, x_test, y_train, y_test = train_test_split(x_cheating, np.ravel(y), test_size=0.2, random_state=42)

model = LogisticRegression(solver='liblinear', max_iter=1000)
model.fit(x_train, y_train)

predictions = model.predict(x_test)
print(f"Accuracy score: {accuracy_score(y_test, predictions):.2f}")

Accuracy score: 0.79
